# Notebook 5: Train a Mini GPT on Running Data

**Goal**: Train a small language model end-to-end and watch the loss decrease.
Observe training dynamics: loss curves, perplexity, learning rate schedule.

**Expected time**: ~10–30 minutes on CPU for 1000 steps.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import math
import matplotlib.pyplot as plt

from 01_data.generate_synthetic import SyntheticRunGenerator
from 01_data.data_loader import RunningDataset, build_dataloaders
from 02_tokenization.bpe_tokenizer import BPETokenizer
from 05_transformer.gpt_model import RunningGPT
from 06_training.optimizer import build_optimizer, CosineWarmupScheduler
from 06_training.loss import cross_entropy_loss, perplexity

torch.manual_seed(42)
device = torch.device('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                      else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Prepare data and tokenizer

In [ ]:
gen = SyntheticRunGenerator(seed=42)
corpus = gen.generate_run_notes(3000)

print('Training BPE tokenizer...')
tok = BPETokenizer(vocab_size=1000)
tok.train(corpus, verbose=False)
print(f'Vocab size: {len(tok):,}')

# Encode entire corpus to a flat token sequence
all_ids = []
for doc in corpus:
    all_ids.extend(tok.encode(doc, add_special=True))
    
print(f'Total tokens: {len(all_ids):,}')

In [ ]:
CONTEXT_LEN = 64   # shorter for fast demo; use 256 for real training

dataset = RunningDataset(all_ids, context_len=CONTEXT_LEN)
train_loader, val_loader = build_dataloaders(dataset, batch_size=16)
print(f'Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}')

## 2. Build the model

In [ ]:
model = RunningGPT(
    vocab_size=len(tok),
    d_model=64,
    num_heads=2,
    num_layers=2,
    d_ff=128,
    max_seq_len=CONTEXT_LEN,
    dropout=0.1,
).to(device)

counts = model.count_parameters()
print(f'\nParameter breakdown:')
for k, v in counts.items():
    print(f'  {k:<25}: {v:>10,}')

## 3. Training loop with live loss tracking

In [ ]:
MAX_STEPS = 500   # increase to 5000 for real training
LR = 3e-4
WARMUP = 50

optimizer = build_optimizer(model, lr=LR)
scheduler = CosineWarmupScheduler(optimizer, warmup_steps=WARMUP, max_steps=MAX_STEPS)

train_losses = []
val_losses = []
lr_history = []

data_iter = iter(train_loader)
model.train()
optimizer.zero_grad()

for step in range(1, MAX_STEPS + 1):
    try:
        x, y = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x, y = next(data_iter)
    
    x, y = x.to(device), y.to(device)
    logits, _ = model(x)
    loss = cross_entropy_loss(logits, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()
    
    lr_history.append(scheduler.get_last_lr()[0])
    train_losses.append(loss.item())
    
    if step % 100 == 0:
        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = sum(cross_entropy_loss(model(xv.to(device))[0], yv.to(device)).item()
                         for xv, yv in val_loader) / len(val_loader)
        val_losses.append((step, val_loss))
        model.train()
        print(f'Step {step:5d} | train_loss={loss.item():.4f} | val_loss={val_loss:.4f} | ppl={math.exp(val_loss):.1f} | lr={scheduler.get_last_lr()[0]:.2e}')

print('\nTraining complete!')

## 4. Plot training dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Smooth training loss
window = 20
smoothed = [sum(train_losses[max(0,i-window):i+1])/min(i+1,window) for i in range(len(train_losses))]
axes[0].plot(smoothed, color='steelblue', linewidth=1.5)
if val_losses:
    axes[0].scatter(*zip(*val_losses), color='coral', s=40, zorder=5, label='val loss')
    axes[0].legend()
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training loss curve')
axes[0].grid(True, alpha=0.3)

# Perplexity
axes[1].plot([math.exp(l) for l in smoothed], color='steelblue')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity (lower = better)')
axes[1].grid(True, alpha=0.3)

# LR schedule
axes[2].plot(lr_history, color='coral')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Learning rate')
axes[2].set_title('LR schedule (warmup + cosine)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Generate text from the trained model

In [ ]:
from 07_inference.generate import generate

model.eval()
prompts = [
    'easy run today',
    'marathon training',
    'tempo run 8 miles',
]

for prompt in prompts:
    ids = tok.encode(prompt)
    x = torch.tensor(ids).unsqueeze(0).to(device)
    out = generate(model, x, max_new_tokens=30, temperature=0.8, top_p=0.9, eos_id=tok.eos_id)
    new_ids = out[0, len(ids):].tolist()
    print(f'Prompt: "{prompt}"')
    print(f'Output: "{prompt} {tok.decode(new_ids)}"')
    print()

## Exercise

1. The model trained for only 500 steps. Set `MAX_STEPS=2000`. How much does perplexity improve?
2. Try `temperature=0.3` (conservative) vs `temperature=1.5` (creative). How does output quality change?
3. Double `d_model` to 128. Does it train faster or slower? Does perplexity improve?
4. Remove the LR warmup (set `WARMUP=0`). Does training become unstable? Check the loss curve.